In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# รัน dynamic circuits

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<Accordion>
<AccordionItem title="เวอร์ชันแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Dynamic circuits เป็นเครื่องมือที่ทรงพลังที่คุณสามารถวัด qubits ระหว่างการรัน quantum circuit และดำเนินการ logic แบบ classical ภายใน circuit ตามผลของการวัดระหว่าง circuit เหล่านั้น กระบวนการนี้เรียกอีกอย่างว่า _classical feedforward_ แม้ว่านี่จะเป็นช่วงแรกของการทำความเข้าใจวิธีใช้ประโยชน์จาก dynamic circuits ชุมชนวิจัย quantum ได้ระบุกรณีการใช้งานหลายอย่าง เช่น:

* การเตรียม quantum state ที่มีประสิทธิภาพ เช่น [GHZ state](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [W-state](https://arxiv.org/abs/2403.07604) (สำหรับข้อมูลเพิ่มเติมเกี่ยวกับ W-state ดูที่ ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) และ [matrix product states](https://arxiv.org/abs/2404.16083) ประเภทกว้างๆ
* [Long-range entanglement ที่มีประสิทธิภาพ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) ระหว่าง qubits บน chip เดียวกันโดยใช้ shallow circuits
* [การสุ่มตัวอย่าง IQP-like circuits](https://arxiv.org/pdf/2505.04705) ที่มีประสิทธิภาพ

อย่างไรก็ตาม การปรับปรุงเหล่านี้ที่ dynamic circuits นำมามาพร้อมกับการแลกเปลี่ยน การวัดระหว่าง circuit และ classical operations โดยทั่วไปมีเวลาดำเนินการนานกว่า two-qubit gates และการเพิ่มขึ้นของเวลานี้อาจทำให้ประโยชน์ของ circuit depth ที่ลดลงหายไป ดังนั้น การลดความยาวของการวัดระหว่าง circuit จึงเป็นพื้นที่เน้นของการปรับปรุงเมื่อ IBM Quantum&reg; เผยแพร่ [เวอร์ชันใหม่](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) ของ dynamic circuits สำหรับข้อจำกัดอื่นๆ เมื่อใช้ dynamic circuits ดู [Feature compatibility table](/guides/estimator-options#options-compatibility-table) ของ Estimator หรือ [Sampler](/guides/sampler-options#options-compatibility-table)

[OpenQASM 3 specification](https://openqasm.com/language/classical.html#looping-and-branching) กำหนดโครงสร้าง control-flow หลายอย่าง แต่ Qiskit Runtime รองรับเฉพาะคำสั่ง conditional `if` เท่านั้น ใน Qiskit SDK สิ่งนี้สอดคล้องกับเมธอด [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) บน [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) เมธอดนี้ส่งคืน [context manager](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) และโดยทั่วไปใช้ใน `with` statement คู่มือนี้อธิบายวิธีใช้ conditional statement นี้

> **Note:** ตัวอย่างโค้ดในคู่มือนี้ใช้คำสั่ง measure มาตรฐานสำหรับการวัดระหว่าง circuit อย่างไรก็ตาม แนะนำให้ใช้คำสั่ง `MidCircuitMeasure` แทน ถ้า backend รองรับ ดูรายละเอียดในส่วน [Mid-circuit measurements](#midcircuit)
## หา backends ที่รองรับ dynamic circuits
เพื่อหา backends ทั้งหมดที่บัญชีของคุณสามารถเข้าถึงและรองรับ dynamic circuits ให้รันโค้ดเช่นต่อไปนี้ ตัวอย่างนี้สมมติว่าคุณได้ [บันทึก credentials](/guides/save-credentials) แล้ว คุณยังสามารถ [ระบุ credentials อย่างชัดเจน](/guides/initialize-account#explicit) เมื่อ initialize Qiskit Runtime service account ของคุณ ซึ่งจะให้คุณดู backends ที่มีบน instance หรือประเภท plan เฉพาะได้

> **Note:** - Backends ที่มีสำหรับบัญชีขึ้นอยู่กับ instance ที่ระบุใน credentials
> - เวอร์ชันใหม่ของ dynamic circuits ตอนนี้มีให้สำหรับผู้ใช้ทั้งหมดบน backends ทั้งหมด ดู [ประกาศ](/announcements/product-updates/2025-09-25-new-dynamic-circuits) สำหรับรายละเอียดเพิ่มเติม

In [1]:
# This cell is hidden from users.  It hides all those "...instance was not set..." warnings.
import warnings

warnings.filterwarnings("ignore", message=".*Instance was not set*")

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_boston')>, <IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


<span id="midcircuit"></span>
## Mid-circuit measurements

Prior to `qiskit-ibm-runtime` v0.43.0, `measure` was the only measurement instruction in Qiskit. Mid-circuit measurements, however, have different tuning requirements than _terminal_ measurements (measurements that happen at the end of a circuit). For example, you need to consider the instruction duration when tuning a mid-circuit measurement because longer instructions cause noisier circuits. You don't need to consider instruction duration for terminal measurements because there are no instructions after terminal measurements.

<Admonition type="note">
The `MidCircuitMeasure` instruction maps to the `measure_2` instruction reported in the backend's `supported_instructions`. However,  `measure_2` is not supported on all backends. Use `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` to find backends that support it.  New measurements might be added in the future, but this is not guarenteed.
</Admonition>

### `MidCircuitMeasure` method

In `qiskit-ibm-runtime` v0.43.0, the `MidCircuitMeasure` instruction was introduced. As the name suggests, it is a new measurement instruction that is optimized for mid-circuit on IBM&reg; QPUs.  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [3]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


<span id="midcircuit"></span>
## Mid-circuit measurements

ก่อน `qiskit-ibm-runtime` v0.43.0 `measure` เป็นคำสั่งการวัดเดียวใน Qiskit อย่างไรก็ตาม mid-circuit measurements มีข้อกำหนดการปรับแต่งที่แตกต่างจากการวัด _terminal_ (การวัดที่เกิดขึ้นตอนท้ายของ circuit) ตัวอย่างเช่น คุณต้องพิจารณา instruction duration เมื่อปรับแต่ง mid-circuit measurement เพราะ instructions ที่ยาวกว่าทำให้ circuits มี noise มากขึ้น คุณไม่จำเป็นต้องพิจารณา instruction duration สำหรับ terminal measurements เนื่องจากไม่มี instructions หลัง terminal measurements

> **Note:** คำสั่ง `MidCircuitMeasure` map ไปยังคำสั่ง `measure_2` ที่รายงานใน `supported_instructions` ของ backend อย่างไรก็ตาม `measure_2` ไม่รองรับบน backends ทั้งหมด ใช้ `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` เพื่อหา backends ที่รองรับ การวัดใหม่อาจเพิ่มในอนาคต แต่ไม่รับประกัน

### เมธอด `MidCircuitMeasure`
ใน `qiskit-ibm-runtime` v0.43.0 มีการนำคำสั่ง `MidCircuitMeasure` มาใช้ ตามชื่อที่บ่งบอก เป็นคำสั่งการวัดใหม่ที่ optimize สำหรับ mid-circuit บน IBM&reg; QPUs แม้ว่าคุณสามารถใช้ `QuantumCircuit.measure` สำหรับ mid-circuit measurement แต่ด้วยการออกแบบ `MidCircuitMeasure` โดยทั่วไปเป็นตัวเลือกที่ดีกว่า ตัวอย่างเช่น มี overhead น้อยกว่าใน circuit ของคุณกว่าเมื่อใช้ `QuantumCircuit.measure`

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)


# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d82864vtjchs73bnokf0 (DONE)


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use [Sampler](/docs/guides/get-started-with-sampler) and build your own measurement circuits instead.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting), for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility [`measurement_bases` module](https://qiskit.github.io/qiskit-addon-utils/apidocs/qiskit_addon_utils.exp_vals.html#qiskit_addon_utils.exp_vals.get_measurement_bases) for more information. For more information, see the [documentation](https://qiskit.github.io/qiskit-addon-utils/) for the Qiskit addon utilities package.
3. Add back together the results for each partition.

## Restrictions

Review any [Feature compatibility table](/docs/guides/estimator-options#options-compatibility-table) to understand restrictions when using dynamic circuits. Note that feature compatibility is not primitive-dependent.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch](/docs/guides/stretch).
- Review the [classical feedforward and control flow](/docs/guides/classical-feedforward-and-control-flow) guide.
- Use [circuit schedule visualization](/docs/guides/qiskit-runtime-circuit-timing) to debug and optimize your dynamic circuits.
</Admonition>